# 17 — Phase 5–5b: Backward in C (MLA → MoE → full block)

**Before:** notebooks 15–16 (PyTorch train + sample; C forward in phase 4).

**Goal:** Mirror `train_gpt2.c` backward — piece by piece in C.

## What exists now

| Piece | Forward | Backward |
|-------|---------|----------|
| RMSNorm | `rmsnorm.c` | `ds4_rmsnorm_backward` |
| MLA | `mla.c` | `dsv2_mla_backward` |
| MoE | `moe.c` + `moe_train.c` | `dsv2_moe_backward` |
| Block | `block_train.c` | `dsv2_block_backward` (both residuals) |
| 1-layer train | `-train-1layer` | MLA + wte (MoE frozen) |
| Full train | `-train-full` | 2 layers, MLA + MoE, B=1, grad clip |

## Try it

```bash
cd c
make test_v2
./bin/train_v2_tiny -train-1layer 30
./bin/train_v2_tiny -train-full 20
```

For production-quality training use notebook 15 `Trainer`, then `scripts/export_v2_tiny.py` and `-sample -ckpt`.


In [ ]:
# PyTorch reference: MLA backward via autograd (same math C implements)
import torch
from llmc.deepseek_v2 import DeepSeekV2Config, MultiHeadLatentAttention

cfg = DeepSeekV2Config.tiny(64, 16)
attn = MultiHeadLatentAttention(cfg)
x = torch.randn(1, 8, cfg.n_embd, requires_grad=True)
y = attn(x)
y.sum().backward()
print("x.grad norm:", x.grad.norm().item())
print("wq.grad norm:", attn.wq.weight.grad.norm().item())


## Read order (C)

1. `deepseek_v2/ops.c` — linear + softmax backward
2. `mla.c` — `dsv2_mla_forward_train` / `dsv2_mla_backward`
3. `moe_train.c` — routed top-k (frozen in backward) + shared experts
4. `block_train.c` — `x = x + attn(ln1(x)); x = x + moe(ln2(x))` (same as PyTorch)
5. `model.c` — `dsv2_model_train_step_full` chains blocks + final RMSNorm + tied wte

**Phase 6 (later):** AdamW, B>1, save C checkpoints after train.
